In [1]:
import cv2
import numpy as np

In [6]:
# 이미지 불러오기
src = cv2.imread("./Data/ticket.png")

gray = cv2.cvtColor(src, cv2.COLOR_BGR2GRAY)

# Prewitt 필터 커널 정의
prewitt_kernel_x = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])
prewitt_kernel_y = np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]])

# Prewitt 필터 적용
# 수평 방향 엣지 검출
prewittX = cv2.filter2D(gray, -1, prewitt_kernel_x)
# 수직 방량 엣지 검출
prewittY = cv2.filter2D(gray, -1, prewitt_kernel_y)

# 엣지 검출 결과의 절대값
abs_prewittX = np.absolute(prewittX)
abs_prewittY = np.absolute(prewittY)

# 엣지 검출 결과를 8-bit 이미지로 변환
prewittX_8u = cv2.normalize(abs_prewittX, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
prewittY_8u = cv2.normalize(abs_prewittY, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)

# 최종 엣지 이미지 생성
edges_prewitt = cv2.bitwise_or(prewittX_8u, prewittY_8u)

cv2.imshow("src", src)
cv2.imshow("Prewitt X", prewittX_8u)
cv2.imshow("Prewitt Y", prewittY_8u)
cv2.imshow("Prewitt Edges", edges_prewitt)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [7]:
# Sobel 필터 적용
sobelX = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3) # 수평
sobelY = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3) # 수직

abs_sobelX = np.absolute(sobelX)
abs_sobelY = np.absolute(sobelY)

sobelX_8u = cv2.normalize(abs_sobelX, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)
sobelY_8u = cv2.normalize(abs_sobelY, None, 0, 255, cv2.NORM_MINMAX, cv2.CV_8U)

edges_sobel = cv2.bitwise_or(sobelX_8u, sobelY_8u)

cv2.imshow("src", src)
cv2.imshow("Sobel X", sobelX_8u)
cv2.imshow("Sobel Y", sobelY_8u)
cv2.imshow("Sobel Edges", edges_sobel)
cv2.imshow("Prewitt Edges", edges_prewitt)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [8]:
# Canny 엣지 검출 적용
canny_edges = cv2.Canny(gray, threshold1=100, threshold2=200)

cv2.imshow("src", src)
cv2.imshow("Canny Edges", canny_edges)
cv2.imshow("Sobel Edges", edges_sobel)
cv2.imshow("Prewitt Edges", edges_prewitt)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [9]:
# 엣지 검출
edges = cv2.Canny(gray, 100, 200, apertureSize=3)

# Hough 변환을 사용한 직선 검출
lines = cv2.HoughLines(edges, 1, np.pi/180, 50)

# 검출된 직선 그리기
for rho, theta in lines[:, 0]:
    a = np.cos(theta)
    b = np.sin(theta)
    x0 = a * rho
    y0 = b * rho
    x1 = int(x0 + 1000 * (-b))
    y1 = int(y0 + 1000 * (a))
    x2 = int(x0 - 1000 * (-b))
    y2 = int(y0 - 1000 * (a))

    cv2.line(src, (x1, y1), (x2, y2), (0, 0, 255), 2)

cv2.imshow("Hough Lines", src)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [10]:
edges = cv2.Canny(gray, 100, 200, apertureSize=3)

# 확률적 Hough 변환을 사용한 직선 검출
linesP = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=50, maxLineGap=10)

# 검출된 직선 그리기
if lines is not None:
    for line in linesP:
        x1, y1, x2, y2 = line[0]
        cv2.line(src, (x1, y1), (x2, y2), (0, 255, 0), 2)

cv2.imshow("Hough LinesP", src)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [11]:
# 노이즈 제거를 위한 가우시안 블러
gray_blurred = cv2.GaussianBlur(gray, (9, 9), 2)

# HoughCircles 함수를 사용하여 원 검출
circles = cv2.HoughCircles(gray_blurred,
                           cv2.HOUGH_GRADIENT,
                           dp = 1,
                           minDist = 20,
                           param1 = 50,
                           param2 = 30,
                           minRadius = 0,
                           maxRadius = 0)

# 원 그리기
if circles is not None:
    circles = np.uint16(np.around(circles))
    for circle in circles[0, :]:
        # 원의 중심 그리기
        cv2.circle(src, (circle[0], circle[1]), 2, (0, 255, 0), -1)
        # 원의 외곽 그리기
        cv2.circle(src, (circle[0], circle[1]), circle[2], (0, 0, 255), 3)

cv2.imshow("Hough Circles", src)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [12]:
img = src.copy()
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

# 윤곽선 찾기
contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)

# 윤곽선 그리기
cv2.drawContours(img, contours, -1, (0, 255, 0), 3)

cv2.imshow("src", src)
cv2.imshow("Contour", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [13]:
# Contour 간략화하여 도형 검출
img2 = src.copy()
gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
edges = cv2.Canny(gray, 30, 200)

contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

for cnt in contours:

    perimeter = cv2.arcLength(cnt, True) # Contour 둘레 길이 연산
    epsilon = perimeter * 0.02
    
    approx = cv2.approxPolyDP(cnt, epsilon, True)

    cv2.drawContours(img2, [approx], 0, (0, 255, 0), 3)

    # 근사화된 점의 개수로 다각형 구별
    if len(approx) == 3:
        cv2.putText(img2,
                    "Triangle",
                     (approx[0][0][0], approx[0][0][1]),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1, (255, 0, 0), 2)
    elif len(approx) == 4:
        cv2.putText(img2,
                    "Rectangle",
                     (approx[0][0][0], approx[0][0][1]),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1, (255, 0, 0), 2)
    else:
        cv2.putText(img2,
                    "Polygon",
                     (approx[0][0][0], approx[0][0][1]),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1, (255, 0, 0), 2)

cv2.imshow("src", src)
cv2.imshow("thresh", thresh)
cv2.imshow("edges", edges)
cv2.imshow("Contour", img2)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [14]:
img3 = src.copy()
gray = cv2.cvtColor(img3, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 30, 200)
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

# 모든 윤곽선의 Convex Hull 계산
convex_hulls = [cv2.convexHull(contour) for contour in contours]

cv2.drawContours(img3, convex_hulls, -1, (0, 0, 0), 3)

cv2.imshow("src", src)
cv2.imshow("Contour", img2)
cv2.imshow("Convex Hull", img3)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [16]:
img4 = src.copy()
gray = cv2.cvtColor(img4, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 30, 200)
contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 객체에 대한 경계 상자, 최소 회전 경계 상자, 최소 경계 원, 타원 찾기
for contour in contours:
    # 경계 상자 찾기
    x, y, w, h = cv2.boundingRect(contour)
    cv2.rectangle(img4, (x, y), (x + w, y + h), (255, 255, 0), 2)

    # 최소 회전 경계 상자 찾기
    rect = cv2.minAreaRect(contour)
    box = cv2.boxPoints(rect)
    box = np.intp(box)
    cv2.drawContours(img4, [box], 0, (0, 150, 255), 2)

    # 최소 경계 원 찾기
    (x, y), radius = cv2.minEnclosingCircle(contour)
    center = (int(x), int(y))
    radius = int(radius)
    cv2.circle(img4, center, radius, (255, 0, 255), 2)

    # 타원 찾기
    if len(contour) >= 5:
        ellipse = cv2.fitEllipse(contour)
        cv2.ellipse(img4, ellipse, (0, 0, 0), 2)

cv2.imshow("Shape Detection", img4)
cv2.waitKey(0)
cv2.destroyAllWindows()